# 🤖 AI Job Market Global 2026 — Starter Notebook

Welcome! This notebook gives you a quick tour of the **AI Job Market Global 2026** dataset:  
5,773 real-time job postings collected from the **Adzuna** and **USAJobs** public APIs across 🇺🇸 🇬🇧 🇨🇦 🇦🇺 🇩🇪.

**What we'll cover:**
1. 📦 Load & inspect the data
2. 💰 Salary landscape by country
3. 🛠️ Most in-demand skills
4. 🌐 Remote vs onsite breakdown
5. 📈 Experience level distribution
6. 🔍 Salary prediction — baseline model

In [ ]:
# ── Install / import ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# ── Load dataset ──────────────────────────────────────────────────────────────
# On Kaggle: BASE points to the dataset directory.
# Locally:   BASE points to the data/ folder.
import os
BASE = '/kaggle/input/ai-job-market-global-2026' if os.path.exists('/kaggle/input') else 'data'
df = pd.read_csv(f'{BASE}/ai_jobs_global.csv', encoding='utf-8', parse_dates=['posted_date'])

print(f"✅ Loaded  {len(df):,} rows  ×  {df.shape[1]} columns")
df.head(3)

## 📦 1. Dataset Overview

In [ ]:
# Quick stats overview
overview = pd.DataFrame({
    'dtype'      : df.dtypes,
    'non_null'   : df.notna().sum(),
    'null_%'     : (df.isna().mean() * 100).round(1),
    'unique'     : df.nunique(),
})

print(f"{'='*50}")
print(f"  Jobs total       : {len(df):,}")
print(f"  Countries        : {df['country'].nunique()}")
print(f"  Unique companies : {df['company'].nunique():,}")
print(f"  Date range       : {df['posted_date'].min().date()}  →  {df['posted_date'].max().date()}")
print(f"  Salary coverage  : {df['salary_min'].notna().mean()*100:.1f}% of rows have salary data")
print(f"{'='*50}")
display(overview)

## 💰 2. Salary Landscape by Country

All salaries are already normalised to **annual USD**. Let's see how compensation compares across markets.

In [ ]:
sal = df[df['salary_min'].notna() & (df['salary_min'] > 0)].copy()

# Summary table
summary = (
    sal.groupby('country')['salary_min']
    .agg(count='count', median='median', mean='mean', p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75))
    .round(0).astype(int)
    .sort_values('median', ascending=False)
    .reset_index()
)
summary.columns = ['Country', 'Jobs w/ Salary', 'Median USD', 'Mean USD', 'P25 USD', 'P75 USD']
display(summary)

# Box plot
fig = px.box(
    sal[sal['country'].isin(summary['Country'])],
    x='country', y='salary_min', color='country',
    color_discrete_sequence=px.colors.qualitative.Bold,
    title='Annual Salary Distribution by Country (USD — Min Reported)',
    labels={'salary_min': 'Salary (USD)', 'country': 'Country'},
    points='outliers',
    category_orders={'country': summary['Country'].tolist()},
)
fig.update_layout(
    showlegend=False, plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee', tickprefix='$', tickformat=',.0f'),
    title_font_size=17,
)
fig.show()

## 🛠️ 3. Most In-Demand Skills

The `required_skills` column contains comma-separated skills extracted from each job description.

In [ ]:
# Flatten comma-separated skills
all_skills = []
for entry in df['required_skills'].dropna():
    all_skills.extend([s.strip() for s in str(entry).split(',') if s.strip()])

skill_counts = pd.Series(Counter(all_skills)).sort_values(ascending=True).tail(20).reset_index()
skill_counts.columns = ['skill', 'count']
skill_counts['pct'] = (skill_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    skill_counts, x='count', y='skill', orientation='h',
    color='count', color_continuous_scale='Blues',
    title='Top 20 In-Demand AI/ML Skills (by job posting count)',
    labels={'count': 'Job Postings', 'skill': ''},
    text=skill_counts['pct'].astype(str) + '%',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white', coloraxis_showscale=False,
    xaxis=dict(gridcolor='#eeeeee'), height=600, title_font_size=17,
)
fig.show()

print("\nTop 5 skills by % of all postings:")
print(skill_counts.tail(5)[['skill','pct']].iloc[::-1].to_string(index=False))

## 🌐 4. Remote vs Hybrid vs Onsite  &  📈 5. Experience Level Distribution

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

PALETTE = px.colors.qualitative.Bold

# ── Remote type pie ──────────────────────────────────────────────────────────
remote_counts = df['remote_type'].value_counts().reset_index()
remote_counts.columns = ['remote_type', 'count']

# ── Experience donut ─────────────────────────────────────────────────────────
exp_order  = ['Junior','Mid-level','Senior','Lead','Management']
exp_counts = df['experience_level'].value_counts().reindex(exp_order).dropna().reset_index()
exp_counts.columns = ['experience_level', 'count']

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'domain'}, {'type': 'domain'}]],
    subplot_titles=['Work Arrangement', 'Experience Level'],
)

fig.add_trace(go.Pie(
    labels=remote_counts['remote_type'], values=remote_counts['count'],
    marker_colors=PALETTE, textinfo='percent+label',
    pull=[0.03]*len(remote_counts), showlegend=False,
), row=1, col=1)

fig.add_trace(go.Pie(
    labels=exp_counts['experience_level'], values=exp_counts['count'],
    marker_colors=PALETTE, textinfo='percent+label', hole=0.45,
    showlegend=False,
), row=1, col=2)

fig.add_annotation(
    text=f"<b>{len(df):,}</b><br>Jobs",
    x=0.78, y=0.5, font_size=14, showarrow=False,
)
fig.update_layout(title_text='Work Arrangement & Seniority Breakdown', title_font_size=17, height=420)
fig.show()

## 💵 6. Salary vs Experience Level

Does seniority actually pay more? Let's check the full distribution, not just the average.

In [ ]:
vdf = df[df['salary_min'].notna() & (df['salary_min'] > 0)].copy()
vdf = vdf[vdf['salary_min'] <= vdf['salary_min'].quantile(0.98)]  # trim extreme outliers
vdf['experience_level'] = pd.Categorical(vdf['experience_level'], categories=exp_order, ordered=True)
vdf = vdf.sort_values('experience_level')

fig = px.violin(
    vdf, x='experience_level', y='salary_min',
    color='experience_level', color_discrete_sequence=PALETTE,
    box=True, points='outliers',
    title='Annual Salary (USD) by Experience Level — Violin + Box',
    labels={'salary_min': 'Min Salary (USD)', 'experience_level': 'Experience Level'},
)
fig.update_layout(
    showlegend=False, plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee', tickprefix='$', tickformat=',.0f'),
    title_font_size=17,
)
fig.show()

# Stats table
stats = (
    vdf.groupby('experience_level', observed=True)['salary_min']
    .agg(['count','median','mean'])
    .rename(columns={'count':'n','median':'Median USD','mean':'Mean USD'})
    .round(0).astype({'n':int,'Median USD':int,'Mean USD':int})
)
display(stats)

## 🤖 7. Salary Prediction — Baseline Model

A quick **Random Forest regressor** predicting `salary_min` from:
- `country`, `experience_level`, `remote_type` (label-encoded)
- `skill_count` — number of skills listed in the posting

This is intentionally simple. Can **you** beat this baseline? 🏆

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# ── Feature engineering ───────────────────────────────────────────────────────
mdf = df[df['salary_min'].notna() & (df['salary_min'] > 0)].copy()
mdf = mdf[mdf['salary_min'] <= mdf['salary_min'].quantile(0.98)]

# Number of skills as a numeric feature
mdf['skill_count'] = mdf['required_skills'].fillna('').apply(
    lambda x: len([s for s in x.split(',') if s.strip()])
)

# Label-encode categorical features
cat_cols = ['country', 'experience_level', 'remote_type']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    mdf[col + '_enc'] = le.fit_transform(mdf[col].fillna('Unknown'))
    encoders[col] = le

feature_cols = [c + '_enc' for c in cat_cols] + ['skill_count']
X = mdf[feature_cols]
y = mdf['salary_min']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Train ─────────────────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
preds = rf.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2  = r2_score(y_test, preds)

print(f"{'='*40}")
print(f"  Baseline Random Forest Results")
print(f"  Training samples : {len(X_train):,}")
print(f"  Test samples     : {len(X_test):,}")
print(f"  MAE              : ${mae:,.0f}")
print(f"  R²               : {r2:.3f}")
print(f"{'='*40}")

# ── Feature importance ────────────────────────────────────────────────────────
fimp = pd.DataFrame({
    'feature': ['Country','Experience Level','Remote Type','Skill Count'],
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(
    fimp, x='importance', y='feature', orientation='h',
    color='importance', color_continuous_scale='Teal',
    title=f'Feature Importance — RF Baseline  (R² = {r2:.3f})',
    labels={'importance': 'Importance Score', 'feature': ''},
    text=fimp['importance'].round(3),
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white', coloraxis_showscale=False,
    xaxis=dict(gridcolor='#eeeeee'), title_font_size=17, height=320,
)
fig.show()

---

## 🚀 What's Next — Ideas for Your Own Notebook

| Idea | Difficulty |
|---|---|
| Add TF-IDF / sentence embeddings on `job_description` for better salary prediction | ⭐⭐ |
| Build a job role classifier from raw descriptions (NLP) | ⭐⭐ |
| Analyse which skills command the highest salary premium | ⭐ |
| Cluster job postings by skill fingerprint using K-Means | ⭐⭐ |
| Time-series analysis of new postings per week | ⭐ |
| Compare private sector (Adzuna) vs government (USAJobs) roles | ⭐ |
| Build a Gradio / Streamlit salary estimator | ⭐⭐⭐ |

---

> 💬 **Found this useful? Please upvote the dataset and drop a comment!**  
> 🍴 The full collection pipeline is open source → [github.com/mercydeez/ai-job-market-dataset](https://github.com/mercydeez/ai-job-market-dataset)